# Guide: Short-Rate Models and FRA Convexity

Runs Ho-Lee and Hull-White FRA analytics with deterministic seeds and compares baseline vs stressed volatility regimes.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().resolve().parent))

from src.models.short_rate import HoLeeModel, HullWhite1FModel, simulate_fra_distribution, convexity_adjustment_summary


In [ ]:
curve = pd.DataFrame({
    't': np.linspace(1/12, 5.0, 60),
    'zero_rate': np.full(60, 0.045),
})

ho = HoLeeModel(sigma=0.01)
hw = HullWhite1FModel(a=0.25, sigma=0.01)

fra_ho = simulate_fra_distribution(ho, curve, start=0.5, end=1.0, n_paths=5000, seed=101)
fra_hw = simulate_fra_distribution(hw, curve, start=0.5, end=1.0, n_paths=5000, seed=101)

summary = pd.DataFrame({
    'model': ['Ho-Lee','Hull-White'],
    'mean_pnl': [fra_ho.pnl.mean(), fra_hw.pnl.mean()],
    'p05_pnl': [np.quantile(fra_ho.pnl,0.05), np.quantile(fra_hw.pnl,0.05)],
    'p95_pnl': [np.quantile(fra_ho.pnl,0.95), np.quantile(fra_hw.pnl,0.95)],
    'convexity_bp_proxy': [1e4*(fra_ho.futures_rate.mean()-fra_ho.fra_forward.mean()), 1e4*(fra_hw.futures_rate.mean()-fra_hw.fra_forward.mean())],
})
summary


In [ ]:
# Regime shift: higher volatility state.
conv_table = convexity_adjustment_summary(
    model=HoLeeModel(sigma=0.01),
    curve=curve,
    tenors=[(0.25,0.5),(0.5,1.0),(1.0,2.0)],
    vol_regimes=[0.01, 0.025],
    n_paths=6000,
    seed=202,
)
conv_table


In [ ]:
class ShortRateExplainer:
    def __init__(self, summary: pd.DataFrame, conv_table: pd.DataFrame):
        self.summary = summary
        self.conv_table = conv_table

    def render_full_markdown(self) -> str:
        stressed = self.conv_table[self.conv_table['vol_regime']==0.025]['convexity_adjustment'].mean()*1e4
        base = self.conv_table[self.conv_table['vol_regime']==0.01]['convexity_adjustment'].mean()*1e4
        delta = stressed - base
        return f"""
### Interpretation (P&L / DV01 / Convexity / Basis / Hedge Action)
- **Regime shift:** moving from low vol to high vol raises average convexity adjustment by about **{delta:.2f} bp**.
- **P&L distribution:** stressed volatility widens FRA P&L tails, so 5th percentile losses deepen.
- **DV01 lens:** unchanged notionals with same DV01 still face larger nonlinear risk in high-vol states.
- **Convexity:** futures-vs-FRA wedge expands in high-vol regime.
- **Basis angle:** if basis and rates vol jump together, isolated FRA hedges can miss funding-driven P&L.
- **Hedge action:** pair DV01-neutral rates hedges with optionality or dynamic convexity overlays in volatile regimes.
""".strip()

explainer = ShortRateExplainer(summary, conv_table)
print(explainer.render_full_markdown())
